In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow.keras import layers, models, optimizers, initializers

np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Flatten
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# One-hot encoding
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
def create_mlp(weight_init, optimizer):
    model = models.Sequential([
        layers.Dense(128, activation='relu',
                     kernel_initializer=weight_init,
                     input_shape=(784,)),
        layers.Dense(64, activation='relu',
                     kernel_initializer=weight_init),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model



In [ ]:
EPOCHS = 10          # was 20
BATCH_SIZE = 256     # was 128


In [ ]:

init_random = initializers.RandomNormal(stddev=0.05)
init_xavier = initializers.GlorotUniform()
init_he = initializers.HeNormal()

model_random = create_mlp(init_random, optimizers.Adam())
history_random = model_random.fit(
    x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, verbose=0)

model_xavier = create_mlp(init_xavier, optimizers.Adam())
history_xavier = model_xavier.fit(
    x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, verbose=0)

model_he = create_mlp(init_he, optimizers.Adam())
history_he = model_he.fit(
    x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, verbose=0)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model_he_sgd = create_mlp(init_he, optimizers.SGD(learning_rate=0.01))
history_he_sgd = model_he_sgd.fit(
    x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, verbose=0)


In [ ]:
def plot_training_curves(histories, labels):
    plt.figure(figsize=(14,5))

    # Loss
    plt.subplot(1,2,1)
    for h, label in zip(histories, labels):
        plt.plot(h.history['loss'], label=label)
    plt.xlabel("Epochs")
    plt.ylabel("Training Loss")
    plt.title("Training Loss vs Epochs")
    plt.legend()
    plt.grid()

    # Accuracy
    plt.subplot(1,2,2)
    for h, label in zip(histories, labels):
        plt.plot(h.history['accuracy'], label=label)
    plt.xlabel("Epochs")
    plt.ylabel("Training Accuracy")
    plt.title("Training Accuracy vs Epochs")
    plt.legend()
    plt.grid()

    plt.show()


In [ ]:
histories = [
    history_random,
    history_xavier,
    history_he,
    history_he_sgd
]

labels = [
    "Random + Adam",
    "Xavier + Adam",
    "He + Adam",
    "He + SGD"
]

plot_training_curves(histories, labels)


In [ ]:
accuracy_table = {
    "Configuration": [
        "Random + Adam",
        "Xavier + Adam",
        "He + Adam",
        "He + SGD"
    ],
    "Final Training Accuracy": [
        history_random.history['accuracy'][-1],
        history_xavier.history['accuracy'][-1],
        history_he.history['accuracy'][-1],
        history_he_sgd.history['accuracy'][-1]
    ],
    "Final Test Accuracy": [
        model_random.evaluate(x_test, y_test, verbose=0)[1],
        model_xavier.evaluate(x_test, y_test, verbose=0)[1],
        model_he.evaluate(x_test, y_test, verbose=0)[1],
        model_he_sgd.evaluate(x_test, y_test, verbose=0)[1]
    ]
}

df_accuracy = pd.DataFrame(accuracy_table)
print(df_accuracy)


   Configuration  Final Training Accuracy  Final Test Accuracy
0  Random + Adam                 0.989854               0.9724
1  Xavier + Adam                 0.992125               0.9725
2      He + Adam                 0.992958               0.9729
3       He + SGD                 0.911958               0.9195
